In [1]:
!pip install -q transformers accelerate datasets huggingface_hub

In [ ]:
from huggingface_hub import login


In [3]:
!git clone https://github.com/nishkalavallabhi/OneStopEnglishCorpus.git

Cloning into 'OneStopEnglishCorpus'...
remote: Enumerating objects: 3074, done.
remote: Total 3074 (delta 0), reused 0 (delta 0), pack-reused 3074 (from 1)
Receiving objects: 100% (3074/3074), 23.50 MiB | 12.77 MiB/s, done.
Resolving deltas: 100% (1143/1143), done.
Updating files: 100% (2656/2656), done.


In [4]:
import json

input_file = "OneStopEnglishCorpus/Sentence-Aligned/ADV-ELE.txt"
output_file = "train.jsonl"

examples = []

with open(input_file, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines()]

i = 0
while i < len(lines):
    if lines[i] == "" or lines[i] == "*******":
        i += 1
        continue

    advanced = lines[i]
    simple = lines[i + 1]

    examples.append({
        "instruction": (
            "Simplify the following English sentence while preserving its original meaning. "
            "Use clear, simple vocabulary and grammar. "
            "Do not add, remove, or change any information."
        ),
        "input": advanced,
        "output": simple
    })

    i += 2

with open(output_file, "w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")

print(f"Wrote {len(examples)} examples")


Wrote 2166 examples


In [5]:
!ls

OneStopEnglishCorpus  sample_data  train.jsonl


In [6]:
!rm -rf OneStopEnglishCorpus
!git clone https://github.com/nishkalavallabhi/OneStopEnglishCorpus.git

Cloning into 'OneStopEnglishCorpus'...
remote: Enumerating objects: 3074, done.
remote: Total 3074 (delta 0), reused 0 (delta 0), pack-reused 3074 (from 1)
Receiving objects: 100% (3074/3074), 23.50 MiB | 12.04 MiB/s, done.
Resolving deltas: 100% (1143/1143), done.
Updating files: 100% (2656/2656), done.


In [7]:
!find OneStopEnglishCorpus -name "*.txt"

OneStopEnglishCorpus/Sentence-Aligned/ADV-ELE.txt
OneStopEnglishCorpus/Sentence-Aligned/ELE-INT.txt
OneStopEnglishCorpus/Sentence-Aligned/ADV-INT.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Experience-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL The millenials-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/Gorillas and oil-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Bogus Allergy-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Ships noise-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Revolution-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Making a killing-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/Lie detector-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Five jobs-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Fifth of young adults-adv.txt
OneSto

In [8]:
!find OneStopEnglishCorpus -name "*.txt"

OneStopEnglishCorpus/Sentence-Aligned/ADV-ELE.txt
OneStopEnglishCorpus/Sentence-Aligned/ELE-INT.txt
OneStopEnglishCorpus/Sentence-Aligned/ADV-INT.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Experience-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL The millenials-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/Gorillas and oil-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Bogus Allergy-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Ships noise-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Revolution-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Making a killing-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/Lie detector-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Five jobs-adv.txt
OneStopEnglishCorpus/Texts-SeparatedByReadingLevel/Adv-Txt/WNL Fifth of young adults-adv.txt
OneSto

In [9]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl")["train"]
print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 2166
})
{'instruction': 'Simplify the following English sentence while preserving its original meaning. Use clear, simple vocabulary and grammar. Do not add, remove, or change any information.', 'input': 'The Seattle-based company has applied for its brand to be a top-level domain name (currently .com), but the South American governments argue this would prevent the use of this internet address for environmental protection, the promotion of indigenous rights and other public interest uses.', 'output': 'Amazon has asked for its company name to be a top-level domain name (currently .com), but the South American governments say this would stop the use of this internet address for environmental protection, indigenous rights and other public interest uses.'}


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
print("Model loaded for zero-shot / few-shot inference")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded for zero-shot / few-shot inference


In [11]:
def zero_shot_generate(sentence, max_new_tokens=60):
    messages = [
        {"role": "system", "content": "You simplify complex English sentences into simple, easy-to-understand English. Reply with only the simplified sentence."},
        {"role": "user", "content": sentence}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [12]:
for ex in dataset.select(range(2)):
    print("Original: ", ex["input"])
    print("Zero-shot:", zero_shot_generate(ex["input"]))
    print("Reference:", ex["output"])
    print("-" * 60)

Original:  The Seattle-based company has applied for its brand to be a top-level domain name (currently .com), but the South American governments argue this would prevent the use of this internet address for environmental protection, the promotion of indigenous rights and other public interest uses.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Zero-shot: The company wants to use a .com domain name, but some countries want to use it for environmental and indigenous rights.
Reference: Amazon has asked for its company name to be a top-level domain name (currently .com), but the South American governments say this would stop the use of this internet address for environmental protection, indigenous rights and other public interest uses.
------------------------------------------------------------
Original:  Until now, the differences between commercial, governmental and other types of identity were easily distinguished in every internet address by .com, .gov and 20 other categories.
Zero-shot: Nowadays, the differences are not easily seen in internet addresses.
Reference: Until now, the differences between commercial, governmental and other types of identity were easy to see in every internet address by the use of .com, .gov and 20 other categories.
------------------------------------------------------------


In [13]:
few_shot_examples = dataset.select(range(3))   # first 3 as demos
test_examples = dataset.select(range(3, 6))    # next 3 for testing, kept separate from demos

In [14]:
FEW_SHOT_SYSTEM_PROMPT = """You simplify complex English sentences into simple, easy-to-understand English.

Simplification requirements:
- Use short, simple sentences.
- Use common, everyday vocabulary; avoid rare, technical, or idiomatic words.
- Preserve the original meaning and all key facts. Do not add or omit information.
- Keep the same overall structure and paragraph order as the source text.
- Do not add a title, commentary, or any text besides the simplified sentence itself.

Below, you are given a few example simplifications. Use them only as a reference
for the style and level of simplicity expected, then simplify the new sentence
the user gives you. Reply with only the simplified sentence, nothing else.
"""

def few_shot_generate(sentence, demos, max_new_tokens=60):
    messages = [
        {"role": "system", "content": FEW_SHOT_SYSTEM_PROMPT}
    ]
    for d in demos:
        messages.append({"role": "user", "content": d["input"]})
        messages.append({"role": "assistant", "content": d["output"]})
    messages.append({"role": "user", "content": sentence})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


In [15]:
for ex in test_examples:
    print("Original: ", ex["input"])
    print("Few-shot: ", few_shot_generate(ex["input"], few_shot_examples))
    print("Reference:", ex["output"])
    print("-" * 60)

Original:  Allowing private companies to register geographical names as gTLDs to reinforce their brand strategy or to profit from the meaning of these names does not serve, in our view, the public interest, the Brazilian Ministry of Science and Technology said.
Few-shot:  Allowing private companies to register geographical names as gTLDs to reinforce their brand strategy or to profit from the meaning of these names does not serve the public interest, the Brazilian Ministry of Science and Technology said.
Reference: Allowing private companies to register geographical names as gTLDs to profit from the meaning of these names is not, in our view, in the public interest, the Brazilian Ministry of Science and Technology said.
------------------------------------------------------------
Original:  Brazil said its views were endorsed last month by other members of the Amazon Cooperation Treaty (Bolivia, Colombia, Ecuador, Guyana, Suriname and Venezuela).
Few-shot:  Brazil said its views were s

In [22]:
import re

COT_SYSTEM_PROMPT = """You simplify complex English sentences into simple, easy-to-understand English.

Simplification requirements:
- Use short, simple sentences.
- Use common, everyday vocabulary; avoid rare, technical, or idiomatic words.
- Preserve the original meaning and all key facts. Do not add or omit information.
- Keep the same overall structure and paragraph order as the source text.
- Do not add a title, commentary, or any text besides the simplified article itself.

Before writing the simplification, think step by step inside <think> tags:
- Identify difficult or advanced vocabulary and what simpler word could replace each one.
- Identify long or complex sentences and how you would break them into shorter ones.
- Note any facts, names, or numbers that must be preserved exactly.
Keep this reasoning brief and internal — it will not be shown to the reader.

Then give the final simplified text inside <simplified> tags, like this:
<think>
...your brief reasoning...
</think>
<simplified>
...the final simplified sentence...
</simplified>
"""

def extract_simplified(raw_text):
    """Pull out the text inside <simplified>...</simplified>.
    Falls back to text after </think>, then to the raw text if no tags are found."""
    match = re.search(r"<simplified>(.*?)</simplified>", raw_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r"</think>(.*)", raw_text, re.DOTALL)
    if match and match.group(1).strip():
        return match.group(1).strip()
    return raw_text.strip()

def cot_generate(sentence, max_new_tokens=200, return_reasoning=False):
    messages = [
        {"role": "system", "content": COT_SYSTEM_PROMPT},
        {"role": "user", "content": sentence}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = output[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    final = extract_simplified(raw_text)
    if return_reasoning:
        return final, raw_text
    return final


In [23]:
for ex in test_examples:
    print("Original:      ", ex["input"])
    print("CoT (zero-shot):", cot_generate(ex["input"]))
    print("Reference:      ", ex["output"])
    print("-" * 60)


Original:       Allowing private companies to register geographical names as gTLDs to reinforce their brand strategy or to profit from the meaning of these names does not serve, in our view, the public interest, the Brazilian Ministry of Science and Technology said.
CoT (zero-shot): think
Allowing private companies to register geographical names as gTLDs is not good for the public interest.
simplified
Private companies should not register geographical names as gTLDs.
Reference:       Allowing private companies to register geographical names as gTLDs to profit from the meaning of these names is not, in our view, in the public interest, the Brazilian Ministry of Science and Technology said.
------------------------------------------------------------
Original:       Brazil said its views were endorsed last month by other members of the Amazon Cooperation Treaty (Bolivia, Colombia, Ecuador, Guyana, Suriname and Venezuela).
CoT (zero-shot): think
Brazil's views were supported by other coun

In [ ]:
results_cot = []
for ex in test_examples:
    results_cot.append({
        "original": ex["input"],
        "reference": ex["output"],
        "zero_shot": zero_shot_generate(ex["input"]),
        "few_shot": few_shot_generate(ex["input"], few_shot_examples),
        "cot": cot_generate(ex["input"]),
    })

df_cot = pd.DataFrame(results_cot)
df_cot.to_csv("zero_few_cot_results.csv", index=False)
df_cot
